# 00 — Bắt đầu với finlens

Notebook này là điều kiện tiên quyết cho mười sáu notebook còn lại. Nó trả lời
năm câu hỏi, theo thứ tự bạn sẽ gặp chúng:

1. Khoá API đặt ở đâu, và cái gì che cái gì?
2. Gói của tôi cho phép làm gì? (`whoami` · `limits`)
3. Lời gọi đầu tiên trả về cái gì?
4. **Đơn vị** — nguồn lỗi số một, và nó sai không kèm một exception nào.
5. Khi hỏng thì hỏng thế nào? (cây ngoại lệ · `on_error` · cache)

Thời gian chạy: dưới một phút.

In [1]:
import sys
from pathlib import Path

# Tìm thư mục gốc repo để import finlens_examples — chạy được dù bạn mở
# notebook từ đâu, không cần pip install -e . cũng không cần PYTHONPATH.
GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import warnings

import pandas as pd

import finlens

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

print("finlens", finlens.build_info()["version"], "· pandas", pd.__version__)

finlens 1.3.0 · pandas 3.0.5


## 1 · Khoá API

Thư viện tìm khoá theo thứ tự ưu tiên từ trên xuống:

| Cách | Viết thế nào | Dùng khi |
|---|---|---|
| Tham số | `finlens.client(api_key="flk_...")` | thử nhanh, script một lần |
| Biến môi trường | `FINLENS_API_KEY=flk_...` | CI, container |
| File cấu hình | `%APPDATA%\finlens\config.toml` | **nhiều notebook trên một máy** |

### Dùng VS Code? Extension FinLens đặt khoá hộ bạn

Extension [FinLens for Visual Studio Code](https://marketplace.visualstudio.com/items?itemName=FinLens.finlens)
cấp và quản lý khoá, không cần vào web. Bốn bước trong Command Palette:

1. `FinLens: Đăng nhập` — email hoặc Google
2. `FinLens: Tạo API key` — khoá được lưu vào **VS Code SecretStorage**
3. ⚠️ `FinLens: Ghi API key ra file cấu hình (cho notebook và terminal ngoài)`
4. Chạy lại cell — **không cần khởi động lại kernel**

**Bước 3 là bắt buộc và đây là lý do:** SecretStorage chỉ extension đọc được.
Kernel Jupyter đang chạy cell này là một **tiến trình Python riêng**, không có
đường nào tới kho bí mật đó. Bỏ qua bước 3 thì `finlens.client()` ném lỗi
`FL_CONFIG` dù bạn vừa tạo khoá xong.

Bước 4 không cần restart vì `finlens.client()` đọc file cấu hình ở **mỗi lần
gọi**, không phải lúc `import finlens`.

⚠️ Đánh đổi: sau bước 3 khoá nằm **plaintext trên đĩa**, mọi tiến trình Python
trên máy đọc được, kể cả sau khi gỡ VS Code. Thu hồi bằng
`FinLens: Xoá API key khỏi file cấu hình`.

### Bẫy che file cấu hình

⚠️ **File đầu tiên tìm thấy là file duy nhất được đọc — không merge.** Một
`./finlens.toml` trong thư mục dự án che *hoàn toàn* file cấu hình máy bạn, kể
cả những khoá nó không khai. Nếu thiếu khoá, thông báo lỗi nói rõ nó đã tìm ở
đâu và file nào che file nào. Trong VS Code, `FinLens: Kiểm tra cấu hình` chẩn
đoán giúp.

Tạo client **không chạm mạng** — đặt ở cell đầu notebook hoặc trong `__init__`
của một lớp đều an toàn.

In [2]:
client = finlens.client()  # khoá lấy từ env hoặc file cấu hình

# Dùng như context manager nếu muốn đóng kết nối gọn gàng:
#   with finlens.client() as client:
#       ...

## 2 · Gói của tôi cho phép làm gì

`whoami()` là lời gọi mạng đầu tiên. Đọc nó *trước* khi thiết kế vòng lặp —
`max_symbols_per_request` và `max_intraday_days` quyết định code của bạn trông
như thế nào.

In [3]:
toi = client.whoami()

print(f"Gói:        {toi['account']['tier']} · {toi['account']['status']}")
print(f"Khoá:       {toi['key']['prefix'][:4]}…  (che bớt — notebook này được commit)")
print(f"Server:     {toi['server_time']}")
print()

han_muc = toi["limits"]
mo_ta = {
    "requests_per_day": "request mỗi ngày",
    "requests_remaining": "còn lại hôm nay",
    "max_symbols_per_request": "mã tối đa mỗi request",
    "max_concurrency": "request song song",
    "max_rows": "dòng tối đa mỗi response",
    "max_intraday_days": "ngày intraday mỗi lời gọi",
    "history_days": "số ngày lịch sử (None = không giới hạn)",
}
for khoa, nhan in mo_ta.items():
    print(f"  {str(han_muc[khoa]):>10}  {nhan}")

Gói:        premium · active
Khoá:       flk_…  (che bớt — notebook này được commit)
Server:     2026-08-11T16:00:25.048490+07:00

       50000  request mỗi ngày
       49129  còn lại hôm nay
         100  mã tối đa mỗi request
          16  request song song
      100000  dòng tối đa mỗi response
          10  ngày intraday mỗi lời gọi
        None  số ngày lịch sử (None = không giới hạn)


Ba con số ở trên đổi cách bạn viết code:

- **`max_symbols_per_request`** — bạn *không* cần tự chia lô. Cứ truyền cả danh
  sách 400 mã; thư viện tự cắt và gọi song song theo `max_concurrency`.
- **`max_intraday_days`** — đây là ràng buộc thật. Lấy intraday một năm phải
  lặp theo cửa sổ; notebook `41` làm mẫu.
- **`max_rows`** — vượt ngưỡng thì `df.attrs["finlens"]["truncated"]` bật lên.
  Kiểm nó, đừng cho rằng frame là đầy đủ.

## 3 · Lời gọi đầu tiên

Mọi phương thức dữ liệu trả về **`pandas.DataFrame` trần** — không wrapper,
không lớp con. Ghi được `to_parquet`, nối được `pd.concat`, dùng được mọi
tutorial pandas bạn từng đọc.

In [4]:
gia = client.eod.stock.ohlcv("HPG,VCB,FPT", start="2026-07-01", end="2026-08-08")

print(type(gia).__name__, gia.shape)
gia.head()

DataFrame (84, 7)


,symbol,date,open,high,low,close,volume
0,FPT,2026-07-01,70.4,73.2,70.3,72.9,11630500.0
1,FPT,2026-07-02,72.9,73.3,72.5,72.5,5690500.0
2,FPT,2026-07-03,72.5,73.1,71.7,72.3,7490500.0
3,FPT,2026-07-06,72.4,74.0,71.9,73.0,11200200.0
4,FPT,2026-07-07,73.0,73.7,72.5,73.2,4557200.0


Ba mã trong **một** lời gọi HTTP, không phải ba. Frame ở dạng **long**: mỗi
dòng là một (mã, phiên), nên `groupby("symbol")` là cách bạn tách chúng ra.

In [5]:
gia.groupby("symbol", observed=True).agg(
    so_phien=("date", "count"),
    gia_dau=("close", "first"),
    gia_cuoi=("close", "last"),
    kl_binh_quan=("volume", "mean"),
).round(2)

,so_phien,gia_dau,gia_cuoi,kl_binh_quan
symbol,,,,
FPT,28,72.90,70.8,7780203.57
HPG,28,23.45,22.0,23936696.43
VCB,28,62.48,59.7,4756704.00


## 4 · Đơn vị — đọc trước khi tính toán

Đây là nguồn lỗi số một khi làm việc với dữ liệu chứng khoán Việt Nam, và nó
**sai âm thầm**: không exception nào, chỉ là một con số sai.

| Loại tài sản | Cột giá | Khối lượng |
|---|---|---|
| Cổ phiếu, ETF, chứng chỉ quỹ | **nghìn VND** (`22.3` = 22.300 đ) | cổ phiếu |
| Chỉ số | điểm chỉ số | cổ phiếu |
| Phái sinh | điểm chỉ số | **hợp đồng** |
| Chứng quyền | **VND thô** | chứng quyền |

Mỗi frame **tự khai** đơn vị của nó. Đừng đoán:

In [6]:
meta = gia.attrs["finlens"]

print("units      ", meta["units"])
print("price_basis", meta["price_basis"])  # 'adjusted' hay 'raw'
print("as_of      ", meta["as_of"])  # mốc nước của dữ liệu
print("truncated  ", meta["truncated"])  # có bị cắt vì max_rows không
print("request_id ", meta["request_id"])  # kèm cái này khi báo lỗi

units       {'symbol': None, 'date': None, 'open': 'kVND', 'high': 'kVND', 'low': 'kVND', 'close': 'kVND', 'volume': 'share'}
price_basis adjusted
as_of       2026-08-11T00:00:00+07:00
truncated   False
request_id  793fef80fc044479ad00e693a7334b6d


### Bẫy: `attrs` không sống sót qua `concat` và `merge`

Đây không phải chuyện lý thuyết. Ghép giá cổ phiếu với giá chứng quyền là một
việc hoàn toàn hợp lý — và nó cho ra một cột `close` trộn hai đơn vị lệch nhau
**1000 lần**, không kèm cảnh báo nào.

In [7]:
hpg = client.eod.stock.ohlcv("HPG", start="2026-08-01")
cw = client.eod.warrant.ohlcv("CHPG2525", start="2026-08-01")

print("Trước khi ghép — đơn vị được khai rõ ràng:")
print(f"  HPG      close = {hpg['close'].iloc[-1]:>8,.1f}  [{hpg.attrs['finlens']['units']['close']}]")
print(f"  CHPG2525 close = {cw['close'].iloc[-1]:>8,.1f}  [{cw.attrs['finlens']['units']['close']}]")

ghep = pd.concat([hpg, cw])
print(f"\nSau pd.concat — attrs: {ghep.attrs}  ← rỗng")
print("\nVà cột close bây giờ là thế này:")
print(ghep.groupby("symbol", observed=True)["close"].last().to_string())

Trước khi ghép — đơn vị được khai rõ ràng:
  HPG      close =     22.1  [kVND]
  CHPG2525 close =  1,630.0  [VND]

Sau pd.concat — attrs: {}  ← rỗng

Và cột close bây giờ là thế này:
symbol
CHPG2525    1630.00
HPG           22.05


`22,1` nằm cạnh `1.630,0` trong cùng một cột. Cả hai đều là số hợp lệ, cả hai
đều đúng — chỉ là chúng đo hai thứ khác nhau. `mean()` trên cột đó ra một con
số vô nghĩa, và không có gì trong frame nói cho bạn biết điều đó.

**Cách sống chung:** đọc đơn vị vào biến *trước* khi ghép.

In [8]:
from finlens_examples import doc_don_vi, nhan_don_vi

don_vi_gia = doc_don_vi(hpg, "close")  # giữ lại trước khi mất
print(f"đơn vị đã giữ: {don_vi_gia!r} → nhãn trục: {nhan_don_vi(hpg, 'close')!r}")

# Nếu thật sự cần một cột chung, quy về VND rồi đặt tên nói rõ điều đó:
hpg_vnd = hpg.assign(close_vnd=hpg["close"] * 1_000)
cw_vnd = cw.assign(close_vnd=cw["close"])
print(pd.concat([hpg_vnd, cw_vnd]).groupby("symbol", observed=True)["close_vnd"].last().to_string())

đơn vị đã giữ: 'kVND' → nhãn trục: 'nghìn VND'
symbol
CHPG2525     1630.0
HPG         22050.0


## 5 · Khi hỏng thì hỏng thế nào

Mọi lỗi kế thừa `finlens.FinLensError` và mang theo `.code`, `.request_id`,
`.doc_url`.

```
FinLensError
├── AuthError          InvalidApiKeyError · ApiKeyExpiredError · AccountExpiredError
├── TierError          DatasetNotInTierError · SymbolNotInTierError
├── QuotaError         RateLimitError · DailyQuotaExceededError
├── ValidationError    InvalidSymbolError · InvalidDateRangeError · InvalidIntervalError
├── TransportError     ConnectionFailedError · TlsVerificationError · RequestTimeoutError
└── DataError          SchemaMismatchError · NoDataError
```

`ValidationError` cũng kế thừa `ValueError`, nên `except ValueError` vẫn bắt được.

### `on_error` — một mã hỏng không làm mất các mã còn lại

Mặc định là `"warn"`: bạn nhận về những mã thành công kèm một cảnh báo.

In [9]:
with warnings.catch_warnings(record=True) as bat_duoc:
    warnings.simplefilter("always")
    ket_qua = client.eod.stock.ohlcv("HPG,KHONGCOMA,VCB", start="2026-08-01")

print(f"Cảnh báo: {bat_duoc[0].category.__name__}")
print(f"  {bat_duoc[0].message}\n")
print(f"Vẫn nhận về {ket_qua['symbol'].nunique()} mã: {sorted(ket_qua['symbol'].unique())}")
print(f"Chi tiết mã hỏng: {ket_qua.attrs['finlens']['failed']}")

Cảnh báo: PartialDataWarning
  1 mã không lấy được: KHONGCOMA. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.

Vẫn nhận về 2 mã: ['HPG', 'VCB']
Chi tiết mã hỏng: {'KHONGCOMA': {'code': 'FL_VALIDATION_SYMBOL', 'message': '[FL_VALIDATION_SYMBOL] Không nhận ra mã này. -> https://docs.finlens.vn/python-sdk/errors/FL_VALIDATION_SYMBOL'}}


Trong pipeline tự động, im lặng đi tiếp là sai. `on_error="raise"` biến nó
thành ngoại lệ — và ngoại lệ đó **vẫn mang theo phần lấy được**, ở `.data`.

In [10]:
try:
    client.eod.stock.ohlcv("HPG,KHONGCOMA,VCB", start="2026-08-01", on_error="raise")
except finlens.PartialFetchError as e:
    print(f"{type(e).__name__}  code={e.code}")
    print(f"  {e}")
    print(f"  .data giữ lại {e.data['symbol'].nunique()} mã lấy được")
    print(f"  .failures: {list(e.failures)}")

PartialFetchError  code=FL_DATA_PARTIAL
  [FL_DATA_PARTIAL] 1 mã không lấy được: KHONGCOMA. Phần lấy được nằm ở `.data`, chi tiết lỗi ở `.failures`. -> https://docs.finlens.vn/python-sdk/errors/FL_DATA_PARTIAL
  .data giữ lại 2 mã lấy được
  .failures: ['KHONGCOMA']


### Khung `try/except` bạn sẽ dùng thật

In [11]:
try:
    df = client.eod.stock.ohlcv("HPG")
except finlens.RateLimitError as e:
    print(f"Chờ {e.retry_after} giây rồi thử lại")
except finlens.DailyQuotaExceededError as e:
    print(f"Hết hạn mức ngày, mở lại lúc {e.resets_at}")
except finlens.InvalidSymbolError:
    print("Mã không tồn tại")
except finlens.FinLensError as e:
    print(f"{e.code}: {e}  (request_id={e.request_id})")
else:
    print(f"OK — {len(df):,} dòng")

OK — 250 dòng


### Ngày thị trường nghỉ: frame rỗng vẫn đúng cột, đúng kiểu

Không `KeyError`, không `None`. Code hạ nguồn chạy tiếp bình thường.

In [12]:
nghi = client.eod.stock.ohlcv("HPG", start="2026-08-09", end="2026-08-09")  # Chủ nhật
print(f"shape={nghi.shape}  ·  cột={list(nghi.columns)}")
print(f"nghi['close'].dtype = {nghi['close'].dtype}  ·  mean() = {nghi['close'].mean()}")

shape=(0, 7)  ·  cột=['symbol', 'date', 'open', 'high', 'low', 'close', 'volume']
nghi['close'].dtype = float64  ·  mean() = nan


## 6 · Cache

Tự động. Dữ liệu lịch sử giữ 7 ngày, phiên gần nhất 60 giây, danh mục và cây
ngành 24 giờ. Lời gọi thứ hai không chạm mạng.

In [13]:
import time

t0 = time.perf_counter()
client.eod.stock.ohlcv("VNM", start="2025-01-01")
lan_1 = time.perf_counter() - t0

t0 = time.perf_counter()
client.eod.stock.ohlcv("VNM", start="2025-01-01")
lan_2 = time.perf_counter() - t0

print(f"lần 1 (mạng):  {lan_1 * 1000:>7,.0f} ms")
print(f"lần 2 (cache): {lan_2 * 1000:>7,.0f} ms   → nhanh gấp {lan_1 / lan_2:,.0f} lần")
print(f"\n{client.cache.stats()}")

lần 1 (mạng):      305 ms
lần 2 (cache):       6 ms   → nhanh gấp 48 lần

{'hits': 1, 'misses': 8, 'entries': 6, 'bytes': 630495, 'evictions': 0}


`refresh=True` bỏ qua cache cho một lời gọi; `client.cache.clear()` xoá sạch.
Trong phiên giao dịch, cache 60 giây nghĩa là số liệu bạn thấy có thể trễ tối
đa một phút — `df.attrs["finlens"]["as_of"]` cho biết mốc nước thật.

## Tổng kết

| Bạn cần | Gọi |
|---|---|
| Kiểm tra gói và hạn mức | `client.whoami()` · `client.limits()` |
| Giá cuối ngày nhiều mã | `client.eod.stock.ohlcv("HPG,VCB")` |
| Đơn vị của một cột | `df.attrs["finlens"]["units"]["close"]` |
| Mốc nước dữ liệu | `df.attrs["finlens"]["as_of"]` |
| Mã nào hỏng trong lời gọi | `df.attrs["finlens"]["failed"]` |
| Biến mã hỏng thành lỗi | `on_error="raise"` |
| Bỏ qua cache | `refresh=True` |
| Đặt khoá từ VS Code | `FinLens: Ghi API key ra file cấu hình` |

**Ba điều mang sang notebook sau:**

1. Đọc `df.attrs["finlens"]["units"]`, đừng đoán. Và đọc nó **trước** `concat`.
2. Không tự chia lô mã — thư viện làm rồi, tốt hơn bạn làm bằng vòng lặp.
3. `on_error="warn"` là mặc định vì nó đúng cho việc khám phá dữ liệu; đổi
   sang `"raise"` khi code chạy tự động.

---

**Tiếp theo:** [`11_gia_va_dien_bien.ipynb`](../01-thi-truong-va-dong-tien/11_gia_va_dien_bien.ipynb) — OHLCV
nhiều mã, `interval`, giá điều chỉnh và cách so hiệu suất giữa các mã.